# Cow Detection + Behavior Classification + BotSort Tracking

Tracking-enhanced variant of the end-to-end pipeline.



In [1]:
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
from ultralytics import YOLO




/home/robin/dev/school/research/cow-sam/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
PREFERRED_YOLO_MODEL_PATH = Path("artifacts/runs/detect/yolo_oneclass/weights/best.pt")
VIT_MODEL_PATH = Path("artifacts/models/cow-behavior-vit")

DETECTION_CONF = 0.25
VIDEO_MAX_FRAMES = 120
TRACKER_CFG = "botsort.yaml"
ARTIFACTS_DIR = Path("artifacts/pipeline_tracking")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

torch.manual_seed(42)




Using CUDA: NVIDIA GeForce RTX 4080


In [3]:
# Load models


def resolve_yolo_model_path(preferred_path: Path) -> Path:
    """Resolve YOLO weights path using preferred path, then latest artifacts fallback."""
    if preferred_path.exists():
        return preferred_path

    candidates = sorted(
        Path("artifacts/runs/detect").rglob("weights/best.pt"),
        key=lambda p: p.stat().st_mtime,
    )
    if candidates:
        latest = candidates[-1]
        print(f"Preferred YOLO path missing. Using latest discovered weights: {latest}")
        return latest

    raise FileNotFoundError(
        "YOLO model not found. Expected either "
        f"{preferred_path} or any artifacts/runs/detect/**/weights/best.pt"
    )


YOLO_MODEL_PATH = resolve_yolo_model_path(PREFERRED_YOLO_MODEL_PATH)

if not VIT_MODEL_PATH.exists():
    raise FileNotFoundError(f"ViT model not found: {VIT_MODEL_PATH}")

detector = YOLO(str(YOLO_MODEL_PATH))
processor = AutoImageProcessor.from_pretrained(str(VIT_MODEL_PATH), use_fast=True)
classifier = AutoModelForImageClassification.from_pretrained(str(VIT_MODEL_PATH)).to(
    device
)
classifier.eval()

id2label = classifier.config.id2label
print(f"Loaded labels: {list(id2label.values())}")




Loading weights: 100%|██████████| 200/200 [00:00<00:00, 549.59it/s, Materializing param=vit.layernorm.weight]                                 


Loaded labels: ['drinking water', 'foraging', 'lying down', 'rumination', 'stand']


In [4]:
def classify_behavior(crop_bgr) -> dict[str, float | str]:
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    inputs = processor(Image.fromarray(crop_rgb), return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = classifier(**inputs).logits
        probs = F.softmax(logits, dim=-1)

    pred_id = int(logits.argmax(-1).item())
    confidence = float(probs[0, pred_id].item())
    return {"class": id2label[pred_id], "conf": confidence}


def detect_and_track(frame_bgr) -> tuple[np.ndarray, np.ndarray | None]:
    """Run YOLO tracking and return xyxy boxes + optional track IDs."""
    result = detector.track(
        source=frame_bgr,
        tracker=TRACKER_CFG,
        conf=DETECTION_CONF,
        persist=True,
        verbose=False,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 4), dtype=int), None

    boxes = result.boxes.xyxy.cpu().numpy().astype(int)
    track_ids = (
        result.boxes.id.cpu().numpy().astype(int)
        if result.boxes.id is not None
        else None
    )
    return boxes, track_ids


def track_color(track_id: int | None) -> tuple[int, int, int]:
    if track_id is None:
        return 0, 255, 0
    rgb = np.random.RandomState(track_id).randint(100, 255, size=3)
    return int(rgb[0]), int(rgb[1]), int(rgb[2])


def process_video_with_tracking(
    video_path: Path,
    output_path: Path,
    max_frames: int = VIDEO_MAX_FRAMES,
) -> dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    stats = {
        "frames": 0,
        "detections": 0,
        "fps": fps,
        "track_timelines": defaultdict(list),
    }

    try:
        while cap.isOpened() and stats["frames"] < max_frames:
            ok, frame = cap.read()
            if not ok:
                break

            boxes, track_ids = detect_and_track(frame)
            for idx, box in enumerate(boxes):
                x1, y1, x2, y2 = box.tolist()
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                behavior = classify_behavior(crop)
                tid = int(track_ids[idx]) if track_ids is not None else None
                if tid is not None:
                    stats["track_timelines"][tid].append(behavior["class"])

                color = track_color(tid)
                label = (
                    f"ID:{tid} | {behavior['class']} ({behavior['conf']:.2f})"
                    if tid is not None
                    else f"{behavior['class']} ({behavior['conf']:.2f})"
                )

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame,
                    label,
                    (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 255, 255),
                    2,
                )

            writer.write(frame)
            stats["detections"] += len(boxes)
            stats["frames"] += 1
    finally:
        cap.release()
        writer.release()

    track_timelines: dict[int, list[str]] = dict(stats["track_timelines"])
    behavior_summary: dict[int, dict[str, dict[str, float]]] = {}
    for tid, timeline in track_timelines.items():
        counts = Counter(timeline)
        total = len(timeline)
        behavior_summary[tid] = {
            behavior: {
                "frames": count,
                "seconds": round(count / fps, 2),
                "percent": round(100.0 * count / total, 2),
            }
            for behavior, count in counts.items()
        }

    return {
        "frames": stats["frames"],
        "detections": stats["detections"],
        "fps": fps,
        "unique_tracks": len(track_timelines),
        "behavior_summary": behavior_summary,
    }




In [5]:
def save_track_behavior_figure(
    behavior_summary: dict[int, dict], output_path: Path
) -> None:
    """Save per-track dominant behavior durations as a bar chart."""
    if not behavior_summary:
        return

    track_ids = sorted(behavior_summary.keys())
    dominant_labels = []
    dominant_seconds = []

    for tid in track_ids:
        behaviors = behavior_summary[tid]
        dominant = max(behaviors.items(), key=lambda item: item[1]["seconds"])
        dominant_labels.append(f"Track {tid}: {dominant[0]}")
        dominant_seconds.append(dominant[1]["seconds"])

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(np.arange(len(track_ids)), dominant_seconds, alpha=0.85)
    ax.set_xticks(np.arange(len(track_ids)))
    ax.set_xticklabels(dominant_labels, rotation=45, ha="right")
    ax.set_ylabel("Seconds")
    ax.set_title("Dominant Behavior Duration per Track")
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)




## Optional Demo Run



In [6]:
videos_dir = Path("data/videos/videos")
if videos_dir.exists():
    videos = list(videos_dir.glob("*.mp4"))
    if videos:
        demo_video = videos[0]
        output_video = ARTIFACTS_DIR / f"tracked_{demo_video.stem}.mp4"

        run_stats = process_video_with_tracking(
            video_path=demo_video,
            output_path=output_video,
            max_frames=VIDEO_MAX_FRAMES,
        )

        print(
            f"Processed {run_stats['frames']} frames | "
            f"detections: {run_stats['detections']} | "
            f"unique tracks: {run_stats['unique_tracks']}"
        )
        print(f"Saved tracked video: {output_video}")

        behavior_plot = ARTIFACTS_DIR / f"track_behavior_summary_{demo_video.stem}.png"
        save_track_behavior_figure(run_stats["behavior_summary"], behavior_plot)
        if behavior_plot.exists():
            print(f"Saved behavior summary figure: {behavior_plot}")


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Resolved 2 packages in 131ms
 Downloaded lap
Prepared 1 package in 54ms
Installed 1 package in 5ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 120 frames | detections: 0 | unique tracks: 0
Saved tracked video: artifacts/pipeline_tracking/tracked_165.mp4
